###  Installation de l'environnement

In [16]:
#%pip install -r requirements.txt

In [17]:
%pip install plotly-express

Note: you may need to restart the kernel to use updated packages.


In [18]:
%pip install tensorflow keras-tuner scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [19]:
%pip install nbformat

Note: you may need to restart the kernel to use updated packages.


### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import plotly.express as px
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.cm as cm

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/Github/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

IOException: IO Error: Could not set lock on file "/home/c-enjalbert/Documents/Github/MSPR/bloc_2/amazing/amazing.duckdb": Conflicting lock is held in /home/c-enjalbert/miniconda3/bin/python3.13 (PID 36830) by user c-enjalbert. See also https://duckdb.org/docs/stable/connect/concurrency

In [22]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 5 tables in the database:

1. all_events
2. loaded_files
3. user_events
4. user_events_11_2019
5. user_segments_kmeans


### Import de la table DuckDB

In [23]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "user_events_11_2019"
TABLE_SEGMENTS = "user_segments_11_2019"
SAMPLE_USER_PERCENT = 0.2
BATCH_SIZE = 1000 

In [24]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ all_events           │
│ loaded_files         │
│ user_events          │
│ user_events_11_2019  │
│ user_segments_kmeans │
└──────────────────────┘



In [25]:
# Examine the all_events table
print(f"First 10 rows of {TABLE_EVENTS} table:")
user_events = con.sql(f"""
    SELECT * FROM {TABLE_EVENTS} ORDER BY RANDOM() LIMIT 30000
""")
user_events.show()
user_events_df = user_events.df()


First 10 rows of user_events_11_2019 table:
┌───────────┬──────────────┬─────────────┬─────────────────┬─────────────────────────┬────────────────────┬────────────────────┬─────────────────────┬──────────────────────┬──────────────────────┬───────────────────────┐
│  user_id  │ total_events │ total_views │ total_purchases │ avg_time_between_events │    total_spent     │     avg_basket     │   last_event_time   │   conversion_rate    │    purchase_ratio    │ days_since_last_event │
│  varchar  │    int64     │   double    │     double      │         double          │       double       │       double       │      timestamp      │        double        │        double        │         int64         │
├───────────┼──────────────┼─────────────┼─────────────────┼─────────────────────────┼────────────────────┼────────────────────┼─────────────────────┼──────────────────────┼──────────────────────┼───────────────────────┤
│ 525069269 │           56 │        54.0 │             0.0 │      24866.

In [26]:
# Drop the datetime column
user_events_df = user_events_df.drop(columns=["last_event_time"])

In [27]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Sélectionner les features numériques uniquement
numeric_features = user_events_df.select_dtypes(include=[np.number])

# Option 1: Standardisation (Z-score normalization) - moyenne=0, écart-type=1
print("Standardisation des données...")
std_scaler = StandardScaler()
X_standardized = std_scaler.fit_transform(numeric_features)

X_num_scaled = X_standardized

Standardisation des données...


In [28]:
# Replace for Using category embeddings
#X_final = np.hstack([X_num_scaled, X_emb_scaled])
X_final = X_num_scaled  # Use scaled numerical features instead of raw dataframe

In [ ]:
# STEP 0: Install necessary packages if not present

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt
import shutil

# Clear previous tuning results to avoid shape mismatch errors
if os.path.exists('tuning_dir'):
    shutil.rmtree('tuning_dir')
    print("Cleared previous tuning results")

X_scaled = X_final 
# 2. DEFINE THE SELF-ADAPTING AUTOENCODER (SAE) - Using Functional API
def build_autoencoder(hp):
    # Input layer
    input_layer = layers.Input(shape=(X_scaled.shape[1],))
    x = input_layer
    
    # Encoder layers - Tune the number of layers and units
    for i in range(hp.Int('num_layers', 1, 3)):
        x = layers.Dense(
            units=hp.Int(f'units_{i}', min_value=16, max_value=32, step=8),
            activation='relu'
        )(x)
    
    # Latent space (The "Compressed" representation)
    latent = layers.Dense(units=hp.Int('latent_dim', 2, 10), activation='relu', name='latent_space')(x)
    
    # Decoder (mirroring the encoder)
    output = layers.Dense(X_scaled.shape[1], activation='linear')(latent)
    
    # Create model using Functional API
    model = keras.Model(inputs=input_layer, outputs=output)
    model.compile(optimizer='adam', loss='mse')
    return model

# 3. TUNE THE AUTOENCODER (Self-Adapting)
tuner = kt.RandomSearch(
    build_autoencoder,
    objective='val_loss',
    max_trials=5,
    executions_per_trial=1,
    directory='tuning_dir',
    project_name='sae_tuning'
)

print("Starting hyperparameter tuning...")
tuner.search(X_scaled, X_scaled, epochs=20, validation_split=0.2, verbose=0)
print("Tuning complete!")

best_model = tuner.get_best_models(num_models=1)[0]

# 4. EXTRACT LATENT SPACE REPRESENTATION
# We create a sub-model that stops at the 'latent_space' layer
encoder = keras.Model(inputs=best_model.input, outputs=best_model.get_layer('latent_space').output)
X_latent = encoder.predict(X_scaled, verbose=0)

print(f"Original Shape: {X_scaled.shape}")
print(f"Latent (Compressed) Shape: {X_latent.shape}")

# 5. ENSEMBLE INTERNAL VALIDATION INDEXES (IVIs)
ks = range(2, 11)
results = {
    'k': [],
    'silhouette': [],
    'calinski': [],
    'davies': []
}

for k in ks:
    kmeans = KMeans(n_clusters=k, random_state=42).fit(X_latent)
    labels = kmeans.labels_
    
    results['k'].append(k)
    results['silhouette'].append(silhouette_score(X_latent, labels))
    results['calinski'].append(calinski_harabasz_score(X_latent, labels))
    results['davies'].append(davies_bouldin_score(X_latent, labels))

res_df = pd.DataFrame(results)

# 6. THE GOATA VOTING SCHEME
# Best k is Max for Silhouette/Calinski, Min for Davies
best_k_si = res_df.loc[res_df['silhouette'].idxmax(), 'k']
best_k_ch = res_df.loc[res_df['calinski'].idxmax(), 'k']
best_k_db = res_df.loc[res_df['davies'].idxmin(), 'k']

votes = [best_k_si, best_k_ch, best_k_db]
final_k = max(set(votes), key=votes.count)

print(f"\n--- ENSEMBLE VOTES ---")
print(f"Silhouette suggests k={best_k_si}")
print(f"Calinski-Harabasz suggests k={best_k_ch}")
print(f"Davies-Bouldin suggests k={best_k_db}")
print(f"WINNING K (Optimal): {final_k}")

# 7. FINAL MODEL EXECUTION
final_kmeans = KMeans(n_clusters=int(final_k), n_init=20, random_state=42)
final_clusters = final_kmeans.fit_predict(X_latent)

# Visualizing results (using first 2 dimensions of latent space)
plt.figure(figsize=(10, 6))
plt.scatter(X_latent[:, 0], X_latent[:, 1], c=final_clusters, cmap='viridis', marker='.')
plt.title(f"Final GOATA Clustering (k={final_k}) in Latent Space")
plt.colorbar(label='Cluster ID')
plt.show()

Cleared previous tuning results


Starting hyperparameter tuning...


In [ ]:
STOP

In [ ]:
# Fermeture de la connexion DuckDB
con.close()